## Compute Global Tunings for all plants in Jardin Fleuris

This script preprocesses raw electrophysiological recordings from multiple plants across different days and seasons.  

## What it does
- Iterates through all **seasons** defined in `JardinFleuris_info`.
- For each day and plant:
  - Loads raw `.raw` files (int16 or float32).
  - Performs **QC checks**:
    - Rejects if too many NaNs, too flat, or empty.
  - Interpolates only **short NaN gaps** (≤ `max_gap_s`).
  - Applies **bandpass filtering** and **notch filters** at mains harmonics.
  - Downsamples deterministically from **128 Hz → 100 Hz**.
  - Normalizes with **z-scoring**.
- Aligns all valid plant signals to the **shortest trace per day**.
- Produces a consistent 3D array:  data_3d.shape = (days, plants, timepoints)
- Saves results in `plant_series_data.npz` containing:
- `data`, `plant_names`, `dates`, `valid_mask`, `serie_name`, `ecosystem`, `target_sf`, `timepoints`.
- Logs QC stats (kept/rejected traces) in `qc_summary.json`.  

In [11]:
# -------- Fast, robust plant pipeline: days × plants × timepoints --------
from scipy.signal import butter, iirnotch, sosfiltfilt, resample_poly, tf2sos
import numpy as np
import os, sys, json

sys.path.append('../scripts')
from JardinFleuris_info import seasons, plant_common_names, serie_code_map

# ==== CONFIG ====
RAW_ROOT      = '../../data/data_jardin_fleuris_raw'
ecosystem     = 'Jardin_Fleuris'
original_sf   = 128
target_sf     = 100
LOGGER_DTYPE  = 'int16'     # 'int16' (scaled to [-1,1]) or 'float32'
INT16_SCALE   = 32768.0

# QC / preprocessing
nan_threshold = 0.10        # reject if >10% NaNs in the raw
max_gap_s     = 8.0         # only interpolate NaN runs <= this
raw_flat_std  = 1e-6        # reject if raw std below this (flat)
post_flat_std = 1e-8        # reject if cleaned std below this

# Filtering (all in ONE sosfiltfilt pass at 128 Hz)
hp_cut   = 0.01             # Hz, high-pass to remove drift; set 0 to disable
lp_cut   = 40.0             # Hz, anti-alias LPF before downsampling
notches  = [9.75, 19.5, 29.3, 39.0]   # essentials only
notch_q  = 30.0             # Q ~ f0/bw; (~1–2 Hz BW around notch)

VERBOSE  = True             # set True for per-plant prints

# ==== UTILITIES ====
def build_sos(fs, hp_cut, lp_cut, notches, notch_q):
    sos_list = []
    if hp_cut and hp_cut > 0:
        sos_list.append(butter(2, hp_cut, btype='highpass', fs=fs, output='sos'))
    if lp_cut and lp_cut < fs/2:
        sos_list.append(butter(4, lp_cut, btype='lowpass', fs=fs, output='sos'))
    for f0 in (notches or []):
        if 0 < f0 < fs/2 - 0.5:
            b, a = iirnotch(f0, Q=notch_q, fs=fs)
            sos_list.append(tf2sos(b, a))
    if not sos_list:
        return None
    return np.vstack(sos_list)

SOS_128 = build_sos(original_sf, hp_cut, lp_cut, notches, notch_q)

def nan_run_lengths(mask):
    if mask.size == 0: return []
    d = np.diff(np.r_[0, mask.view(np.int8), 0])
    starts = np.where(d == 1)[0]
    ends   = np.where(d == -1)[0]
    return list(zip(starts, ends - starts))

def interp_short_gaps(x, fs, max_gap_s):
    """Linear-interpolate NaN runs <= max_gap_s; leave longer runs as NaN."""
    if not np.isnan(x).any():
        return x, 0.0, 0
    x = x.copy()
    mask = np.isnan(x)
    runs = nan_run_lengths(mask)
    too_long = 0
    for s, L in runs:
        if L/fs <= max_gap_s:
            left, right = s-1, s+L
            if left < 0 and right >= x.size:
                return x, mask.mean(), 1
            if left < 0:
                x[s:right] = x[right]
            elif right >= x.size:
                x[s:right] = x[left]
            else:
                x[s:right] = np.linspace(x[left], x[right], L+2)[1:-1]
        else:
            too_long += 1
    return x, mask.mean(), too_long

def resample_128_to_100(x):
    """Deterministic length: floor(N*25/32)."""
    desired = int(np.floor(len(x) * 25 / 32))
    y = resample_poly(x, up=25, down=32)
    if len(y) >= desired:
        return y[:desired]
    return np.pad(y, (0, desired - len(y)), mode='edge')

def read_raw(filepath):
    if LOGGER_DTYPE == 'float32':
        return np.fromfile(filepath, dtype=np.float32)
    return (np.fromfile(filepath, dtype=np.int16).astype(np.float32) / INT16_SCALE)

def clean_one_trace(raw, fs_in=original_sf):
    # --- cheap gates on raw
    if raw.size == 0:
        return None, "empty"
    if not np.isfinite(raw).all():
        raw = raw.copy()
        raw[~np.isfinite(raw)] = np.nan
    if np.isnan(raw).mean() > nan_threshold:
        return None, "too_many_nans"
    if np.nanstd(raw) < raw_flat_std:
        return None, "flat_raw"

    # --- interpolate short gaps only
    x, _, n_long = interp_short_gaps(raw, fs_in, max_gap_s=max_gap_s)
    if n_long > 0:
        return None, f"long_nan_gaps:{n_long}"

    # --- single-pass filtering
    if SOS_128 is not None:
        x = sosfiltfilt(SOS_128, x)

    # --- downsample deterministically
    x = resample_128_to_100(x)

    # --- sanity & standardize (z-score)
    if not np.isfinite(x).all():
        return None, "nonfinite_after_filter"
    sd = float(np.std(x))
    if sd < post_flat_std:
        return None, "flat_after_clean"
    x = (x - float(np.mean(x))) / sd

    return x.astype(np.float32), "ok"

# ==== MAIN ====
for SERIE in seasons:
    code = serie_code_map[SERIE]
    plant_order = list(plant_common_names[code].keys())
    serie_folder = os.path.join(RAW_ROOT, SERIE, 'RAW')
    if not os.path.isdir(serie_folder):
        print(f"[WARN] Missing folder for SERIE={SERIE}: {serie_folder}")
        continue

    dates = sorted(d for d in os.listdir(serie_folder)
                   if os.path.isdir(os.path.join(serie_folder, d)))

    day_arrays, day_valid_masks, kept_dates = [], [], []
    qc_summary = {"kept": 0, "rejected": 0, "reasons": {}}

    for date in dates:
        if VERBOSE:
            print(f'Processing {SERIE} {date}')
        perplant = [None] * len(plant_order)
        valid_mask = np.zeros(len(plant_order), dtype=bool)

        day_path = os.path.join(serie_folder, date)
        for p_idx, plant_name in enumerate(plant_order):
            fp = os.path.join(day_path, f'{plant_name} {date}.raw')
            if not os.path.isfile(fp):
                if VERBOSE: print(f'  {plant_name}: missing')
                qc_summary["rejected"] += 1
                qc_summary["reasons"]["missing"] = qc_summary["reasons"].get("missing", 0) + 1
                continue
            try:
                raw = read_raw(fp)
            except Exception as e:
                if VERBOSE: print(f'  {plant_name}: read error -> {e}')
                qc_summary["rejected"] += 1
                qc_summary["reasons"]["read_error"] = qc_summary["reasons"].get("read_error", 0) + 1
                continue

            cleaned, status = clean_one_trace(raw, original_sf)
            if cleaned is None:
                if VERBOSE: print(f'  {plant_name}: reject ({status})')
                qc_summary["rejected"] += 1
                qc_summary["reasons"][status] = qc_summary["reasons"].get(status, 0) + 1
                continue

            perplant[p_idx] = cleaned
            valid_mask[p_idx] = True
            qc_summary["kept"] += 1
            if VERBOSE: print(f'  {plant_name}: kept {len(cleaned)} @ {target_sf} Hz')

        if not valid_mask.any():
            if VERBOSE: print(f'  {SERIE} {date}: no valid plants, skip day')
            continue

        lengths = [len(ts) for ts in perplant if ts is not None]
        day_min_len = min(lengths)
        day_mat = np.full((len(plant_order), day_min_len), np.nan, dtype=np.float32)
        for p_idx, ts in enumerate(perplant):
            if ts is not None and len(ts) >= day_min_len:
                day_mat[p_idx, :] = ts[:day_min_len]

        day_arrays.append(day_mat)
        day_valid_masks.append(valid_mask)
        kept_dates.append(date)

    if len(day_arrays) == 0:
        print(f'[INFO] No usable days for SERIE={SERIE}')
        continue

    global_min_len = min(arr.shape[1] for arr in day_arrays)
    day_arrays = [arr[:, :global_min_len] for arr in day_arrays]

    data_3d = np.stack(day_arrays, axis=0)            # (days, plants, timepoints)
    valid_mask_2d = np.stack(day_valid_masks, axis=0) # (days, plants)

    print(f'[OK] {SERIE}: data shape = {data_3d.shape} (days, plants, timepoints)')

    out_root = f'../../parcours_musical/plant_data_{ecosystem}/{SERIE}'
    os.makedirs(out_root, exist_ok=True)

    np.savez(
        os.path.join(out_root, 'plant_series_data.npz'),
        data=data_3d,
        plant_names=np.array(plant_order, dtype=object),
        dates=np.array(kept_dates, dtype=object),
        valid_mask=valid_mask_2d,
        serie_name=np.array(SERIE, dtype=object),
        ecosystem=np.array(ecosystem, dtype=object),
        target_sf=np.array(target_sf),
        timepoints=np.array(global_min_len)
    )

    with open(os.path.join(out_root, 'qc_summary.json'), 'w') as f:
        json.dump(qc_summary, f, indent=2)

    print(f'[SAVED] {out_root}/plant_series_data.npz | QC: kept={qc_summary["kept"]}, rejected={qc_summary["rejected"]}')


Processing Serie 7 2024-06-07
  Iris: kept 21447256 @ 100 Hz
  Asphodeline lutea: kept 21447256 @ 100 Hz
  Hemerocallis: kept 21447256 @ 100 Hz
  Lilium candidum: kept 21447256 @ 100 Hz
  Salvia x sylvestris: kept 21447256 @ 100 Hz
  Empty: missing
  Paeonia: kept 21447256 @ 100 Hz
  Lilium henryi var.citrinum: kept 21447256 @ 100 Hz
Processing Serie 7 2024-06-08
  Iris: kept 34559082 @ 100 Hz
  Asphodeline lutea: kept 34559082 @ 100 Hz
  Hemerocallis: kept 34559082 @ 100 Hz
  Lilium candidum: kept 34559082 @ 100 Hz
  Salvia x sylvestris: kept 34559082 @ 100 Hz
  Empty: missing
  Paeonia: kept 34559082 @ 100 Hz
  Lilium henryi var.citrinum: kept 34559082 @ 100 Hz
Processing Serie 7 2024-06-09
  Iris: kept 34559096 @ 100 Hz
  Asphodeline lutea: kept 34559096 @ 100 Hz
  Hemerocallis: kept 34559096 @ 100 Hz
  Lilium candidum: kept 34559096 @ 100 Hz
  Salvia x sylvestris: kept 34559096 @ 100 Hz
  Empty: missing
  Paeonia: kept 34559096 @ 100 Hz
  Lilium henryi var.citrinum: kept 34559096 @

This script analyzes preprocessed plant recordings to extract **spectral peaks**, amplitudes, and musical tunings per day.  

## What it does
- Iterates through all **seasons**.
- Loads each season’s `plant_series_data.npz` (from the preprocessing pipeline).
- For each (day, plant):
- Drops NaNs and very short traces.
- Normalizes the signal and runs **Biotuner** peak extraction.
- Stores up to `n_peaks` **frequency peaks** and **amplitudes**.
- For each **day**:
- Aggregates all plant peaks & amplitudes.
- Computes a **daily tuning** using one of:
  - `"Peaks Ratios"`  
  - `"Dissonance Curve"`  
  - `"Harmonic Fit"`
- Converts tunings into **rational fractions** (max denominator = 100).
- Reduces to a target step size (`num_steps_reduced`) using consonance metrics.
- Saves outputs in `plant_series_peaks.npz` containing:
- `total_peaks`, `total_amps`, `total_tunings`, `total_tunings_reduced`, plus metadata.
- Exports `.scl` **Scala tuning files** (full and reduced tunings) into a `tunings/` subfolder for use in music software.

In [ ]:
import os
from fractions import Fraction
import sys
import numpy as np
import pickle
import pandas as pd
from biotuner.biotuner_object import compute_biotuner
from biotuner.biotuner_utils import compute_peak_ratios, scale2frac, create_SCL
from biotuner.scale_construction import tuning_reduction, diss_curve, harmonic_tuning
from biotuner.peaks_extension import harmonic_fit
from biotuner.metrics import dyad_similarity

# ---- config ----
ecosystem     = 'Jardin_Fleuris'
n_peaks       = 5
peaks_method  = 'harmonic_recurrence'   # or 'EMD'
tuning_method = 'Peaks Ratios'          # 'Peaks Ratios' | 'Dissonance Curve' | 'Harmonic Fit'
min_len_samples = 10_000                # skip traces shorter than this (after NaN drop)
max_denom     = 100                     # for scale2frac
num_steps_reduced = 12                  # target steps in tuning_reduction
SKIP_FIRST    = False                   # if True, drop first returned peak/amp
VERBOSE       = True

sys.path.append('../../scripts')
from utils import average_across_days, average_across_days_phiid
# If you have seasons/serie_code_map etc in scope, import them; else set seasons manually
# from JardinFleuris_info import seasons
# seasons = ['SPRING','SUMMER','FALL','WINTER']  # example
from JardinFleuris_info import seasons  # uses your existing seasons

def pack_fixed(vec, K):
    """Pad/truncate to length K with NaN."""
    v = np.asarray(vec, float)
    out = np.full((K,), np.nan, float)
    k = min(K, v.size)
    if k > 0:
        out[:k] = v[:k]
    return out

def compute_day_tuning(day_peaks, day_amps, method, max_denom, num_steps_reduced):
    """Return (tuning_frac, reduced_tuning_frac). day_* are 1D arrays after NaN masking."""
    if day_peaks.size == 0:
        return np.array([], float), np.array([], float)

    if method == "Peaks Ratios":
        tuning = compute_peak_ratios(day_peaks, rebound=True, sub=False)

    elif method == "Dissonance Curve":
        # upscale peaks for beating model; normalize amps to ~[0.2, 0.8]
        peaks_dc = day_peaks * 128.0
        if day_amps.size and np.nanmax(day_amps) > np.nanmin(day_amps):
            a_min, a_max = float(np.nanmin(day_amps)), float(np.nanmax(day_amps))
            amps_dc = np.interp(day_amps, (a_min, a_max), (0.2, 0.8))
        else:
            amps_dc = np.full_like(day_peaks, 0.5)
        diss, intervals, tuning, *_ = diss_curve(
            peaks_dc, amps_dc,
            denom=max_denom, max_ratio=2,
            euler_comp=False, method='min', plot=False, n_tet_grid=12,
        )

    elif method == "Harmonic Fit":
        # Fit common harmonics to aggregated peaks
        _, harmonics, common_h, _ = harmonic_fit(day_peaks, n_harm=128, bounds=0.1, n_common_harms=2)
        tuning = harmonic_tuning(common_h)

    else:
        raise ValueError(f"Unknown tuning_method: {method}")

    # Clean, unique, to fractions (limit denominators), then reduce to N steps
    tuning = np.round(np.unique(np.asarray(tuning, float)), 5)
    tuning_frac, _, _ = scale2frac(tuning, max_denom)
    tuning_frac = np.unique(tuning_frac)

    # Convert to floats for reduction (keep microtonal ratios faithfully)
    tuning_floats = [float(Fraction(str(x))) for x in tuning_frac]
    # Drop accidental duplicates after float conversion
    tuning_floats = np.unique(np.round(tuning_floats, 8))
    print('LEN TUNING', len(tuning_floats))
    try:
        _, reduced_tuning, _ = tuning_reduction(tuning_floats, num_steps_reduced, function=dyad_similarity)
    except Exception as e:
        print(f"[warn] tuning_reduction failed: {e}")
        reduced_tuning = tuning_floats

    reduced_tuning = np.round(np.unique(reduced_tuning), 5)
    reduced_tuning_frac, _, _ = scale2frac(reduced_tuning, 100)

    return np.asarray(tuning_frac, float), np.asarray(reduced_tuning_frac, float)

def save_scl_set(out_dir, serie_name, day_idx, tuning, reduced):
    """Write two .scl files (full & reduced) appending 2/1 at the top end."""
    os.makedirs(out_dir, exist_ok=True)
    # Ensure 2/1 (octave) is included
    t_full = np.append(tuning, 2.0)
    t_red  = np.append(reduced, 2.0)

    name_full = f"{serie_name}_day_{day_idx}"
    name_red  = f"{serie_name}_day_{day_idx}_reduced"

    scl_full = create_SCL(t_full, name_full, write=False)
    scl_red  = create_SCL(t_red,  name_red, write=False)

    with open(os.path.join(out_dir, f"{name_full}.scl"), "w") as f:
        f.write(scl_full)
    with open(os.path.join(out_dir, f"{name_red}.scl"), "w") as f:
        f.write(scl_red)

# ====================== MAIN: iterate all seasons ======================
for SERIE in seasons:
    serie_name = SERIE
    series_dir = f"../../parcours_musical/plant_data_{ecosystem}/{serie_name}"
    in_path    = os.path.join(series_dir, "plant_series_data.npz")
    if not os.path.isfile(in_path):
        print(f"[WARN] Missing data for {serie_name}: {in_path}")
        continue

    Z = np.load(in_path, allow_pickle=True)
    data_3d    = Z["data"]            # (days, plants, timepoints)
    valid_mask = Z["valid_mask"]      # (days, plants)
    dates      = Z["dates"]
    plant_names = Z["plant_names"]
    target_sf  = int(Z["target_sf"])
    n_days, n_plants, n_timepoints = data_3d.shape

    # Peak/amp containers: fixed (days, plants, n_peaks)
    total_peaks = np.full((n_days, n_plants, n_peaks), np.nan, float)
    total_amps  = np.full((n_days, n_plants, n_peaks), np.nan, float)

    # Day-level tunings: variable length → object arrays
    total_tunings         = np.empty((n_days,), dtype=object)
    total_tunings_reduced = np.empty((n_days,), dtype=object)

    # Nyquist guard
    max_freq = min(45.0, target_sf/2.0 - 1.0)

    print(f"[INFO] {serie_name}: data {data_3d.shape}, fs={target_sf} Hz")

    # ---- per (day, plant): compute peaks/amps
    for d in range(n_days):
        for p in range(n_plants):
            if not valid_mask[d, p]:
                continue

            ts = data_3d[d, p, :]
            ts = ts[~np.isnan(ts)]
            if ts.size < min_len_samples:
                if VERBOSE: print(f"  day {d} plant {p}: too short ({ts.size})")
                continue

            tmin, tmax = float(np.min(ts)), float(np.max(ts))
            if tmax <= tmin:
                continue
            ts = (ts - tmin) / (tmax - tmin)

            try:
                bt = compute_biotuner(
                    sf=target_sf, data=ts,
                    peaks_function=peaks_method, precision=0.01, n_harm=20
                )
                bt.peaks_extraction(
                    min_freq=0.01, max_freq=max_freq,
                    n_peaks=n_peaks, nIMFs=n_peaks
                )
                peaks_arr = np.asarray(bt.peaks, float)
                amps_arr  = np.asarray(bt.amps,  float)
                if SKIP_FIRST and peaks_arr.size:
                    peaks_arr = peaks_arr[1:]
                    amps_arr  = amps_arr[1:]
                # Keep arrays aligned and cap/pad to n_peaks
                L = min(peaks_arr.size, amps_arr.size)
                peaks_arr = peaks_arr[:L]
                amps_arr  = amps_arr[:L]
                total_peaks[d, p, :] = pack_fixed(peaks_arr, n_peaks)
                total_amps[d,  p, :] = pack_fixed(amps_arr,  n_peaks)

                if VERBOSE:
                    kept = np.count_nonzero(~np.isnan(total_peaks[d, p, :]))
                    print(f"  day {d} plant {p}: {kept} peaks")

            except Exception as e:
                print(f"[warn] {serie_name} day={d} plant={p} biotuner failed: {e}")
                continue

        # ---- per day: aggregate and build tuning
        day_peaks = total_peaks[d].reshape(-1)
        day_amps  = total_amps[d].reshape(-1)
        mask = ~np.isnan(day_peaks)
        day_peaks = day_peaks[mask]
        day_amps  = day_amps[mask]

        tuning, reduced = compute_day_tuning(
            day_peaks, day_amps,
            method=tuning_method,
            max_denom=max_denom,
            num_steps_reduced=num_steps_reduced
        )
        total_tunings[d]         = tuning
        total_tunings_reduced[d] = reduced

    # ---- save NPZ per series
    out_npz = os.path.join(series_dir, "plant_series_peaks.npz")
    np.savez(
        out_npz,
        total_peaks=total_peaks,
        total_amps=total_amps,
        total_tunings=total_tunings,                   # object array: per-day 1D arrays
        total_tunings_reduced=total_tunings_reduced,   # object array: per-day 1D arrays
        dates=dates,
        plant_names=plant_names,
        valid_mask=valid_mask,
        n_peaks=np.array(n_peaks),
        peaks_method=np.array(peaks_method, dtype=object),
        tuning_method=np.array(tuning_method, dtype=object),
        sf=np.array(target_sf),
    )
    print(f"[SAVED] {out_npz}")

    # ---- write .scl files per day into tunings/ subfolder
    scl_dir = os.path.join(series_dir, "tunings")
    for d in range(n_days):
        tuning  = total_tunings[d]         if total_tunings[d] is not None else np.array([], float)
        reduced = total_tunings_reduced[d] if total_tunings_reduced[d] is not None else np.array([], float)
        # Skip empty days
        if tuning.size == 0 and reduced.size == 0:
            continue
        save_scl_set(scl_dir, serie_name, d, tuning, reduced)

    print(f"[DONE] {serie_name}: peaks+tunings saved.\n")


[INFO] Serie 7: data (4, 8, 1740800), fs=100 Hz
  day 0 plant 0: 5 peaks
  day 0 plant 1: 5 peaks
  day 0 plant 2: 5 peaks
  day 0 plant 3: 5 peaks
  day 0 plant 4: 5 peaks
  day 0 plant 6: 5 peaks
  day 0 plant 7: 5 peaks
LEN TUNING 171
  day 1 plant 0: 5 peaks
  day 1 plant 1: 5 peaks
  day 1 plant 2: 5 peaks
  day 1 plant 3: 5 peaks
  day 1 plant 4: 5 peaks
  day 1 plant 6: 5 peaks
  day 1 plant 7: 5 peaks
LEN TUNING 177
  day 2 plant 0: 5 peaks
  day 2 plant 1: 5 peaks
  day 2 plant 2: 5 peaks
  day 2 plant 3: 5 peaks
  day 2 plant 4: 5 peaks
  day 2 plant 6: 5 peaks
  day 2 plant 7: 5 peaks
LEN TUNING 196
  day 3 plant 0: 5 peaks
  day 3 plant 1: 5 peaks
  day 3 plant 2: 5 peaks
  day 3 plant 3: 5 peaks
  day 3 plant 4: 5 peaks
  day 3 plant 6: 5 peaks
  day 3 plant 7: 5 peaks
LEN TUNING 238
[SAVED] ../../parcours_musical/plant_data_Jardin_Fleuris/Serie 7\plant_series_peaks.npz
[DONE] Serie 7: peaks+tunings saved.

[INFO] Serie 40: data (4, 8, 1094400), fs=100 Hz
  day 0 plant 0: 